In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

In [2]:
train = pd.read_csv(r"C:\Users\gupta\Downloads\train.csv")
test = pd.read_csv(r"C:\Users\gupta\Downloads\test.csv")

test_ids = test["PassengerId"]
df = pd.concat([train, test], axis=0).reset_index(drop=True)

In [3]:
df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
df.drop(columns=['Cabin'], inplace=True)

df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
df['CabinNumBin'] = pd.qcut(df['CabinNum'], 8, duplicates='drop')

In [4]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

zero_spend = (df[spend_cols].sum(axis=1) == 0)
df.loc[zero_spend & df['CryoSleep'].isna(), 'CryoSleep'] = True

df.loc[df['CryoSleep'] == True, spend_cols] = 0
df['TotalSpend'] = df[spend_cols].sum(axis=1)

In [5]:
df['Group'] = df['PassengerId'].str.split('_').str[0]
df['GroupSize'] = df.groupby('Group')['PassengerId'].transform('count')
df['IsAlone'] = (df['GroupSize'] == 1).astype(int)

In [6]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df['VIP'].fillna(False, inplace=True)
df['CryoSleep'].fillna(False, inplace=True)

for col in ['HomePlanet','Destination','Deck','Side','CabinNumBin']:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in spend_cols + ['TotalSpend']:
    df[col].fillna(0, inplace=True)

C:\Users\gupta\AppData\Local\Temp\ipykernel_9280\3369636339.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\gupta\AppData\Local\Temp\ipykernel_9280\3369636339.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [7]:
for col in spend_cols + ['TotalSpend']:
    df[col] = np.log1p(df[col])

In [8]:
cat_cols = [
    'HomePlanet','CryoSleep','Destination',
    'VIP','Deck','Side','CabinNumBin'
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [9]:
df.drop(columns=['PassengerId','Name','Group'], inplace=True)

X = df.iloc[:len(train)].drop(columns=['Transported'])
y = train['Transported'].astype(int)
X_test = df.iloc[len(train):].drop(columns=['Transported'])

In [10]:
model = CatBoostClassifier(
    iterations=1400,
    depth=8,
    learning_rate=0.02,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=0
)

model.fit(X, y)

In [11]:
preds = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': preds.astype(bool)
})

submission.to_csv("submission.csv", index=False)
print("submission.csv generated 🚀")

submission.csv generated 🚀
